# 1. Data Preprocessing - Smart Greenhouse

This notebook handles data cleaning, normalization, and preparation for ML model training.

## 1. Load and Explore Data

Load sensor logs and explore the dataset structure.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set paths
DATA_PATH = '../../dataset/'
OUTPUT_PATH = '../../dataset/'

# Create sample sensor data if not exists
if not os.path.exists(f'{DATA_PATH}sensor_logs.csv'):
    print("Creating sample sensor data...")
    np.random.seed(42)
    n_samples = 1000
    
    # Generate synthetic sensor data
    timestamps = pd.date_range(start='2024-01-01', periods=n_samples, freq='1H')
    data = {
        'timestamp': timestamps,
        'temperature': np.random.normal(25, 3, n_samples),  # 25°C ± 3
        'humidity': np.random.normal(60, 10, n_samples),    # 60% ± 10
        'co2': np.random.normal(400, 50, n_samples),        # 400ppm ± 50
        'ph': np.random.normal(6.5, 0.5, n_samples),        # pH 6.5 ± 0.5
        'ec': np.random.normal(1.5, 0.3, n_samples)         # EC 1.5 ± 0.3
    }
    
    df = pd.DataFrame(data)
    df.to_csv(f'{DATA_PATH}sensor_logs.csv', index=False)
    print(f"Sample data saved to {DATA_PATH}sensor_logs.csv")
else:
    # Load existing data
    df = pd.read_csv(f'{DATA_PATH}sensor_logs.csv')
    print("Loaded existing sensor data")

print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Explore data
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
df.describe()

## 2. Data Cleaning

Handle missing values and remove outliers.

In [ ]:
# Data Cleaning
df_clean = df.copy()

# Handle missing values - fill with median
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

# Remove outliers using IQR method
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

# Apply to sensor columns
sensor_cols = ['temperature', 'humidity', 'co2', 'ph', 'ec']
for col in sensor_cols:
    df_clean = remove_outliers_iqr(df_clean, col)

print(f"Original rows: {len(df)}")
print(f"After cleaning: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} rows")

## 3. Normalization

Normalize numerical features for better ML model performance.

In [ ]:
# Normalization using Min-Max Scaling
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
sensor_cols = ['temperature', 'humidity', 'co2', 'ph', 'ec']

df_normalized = df_clean.copy()
df_normalized[sensor_cols] = scaler.fit_transform(df_clean[sensor_cols])

print("Normalized data (first 5 rows):")
print(df_normalized[sensor_cols].head())
print("\nNormalized statistics:")
print(df_normalized[sensor_cols].describe())

## 4. Save Processed Data

Save the cleaned and normalized data for training.

In [ ]:
# Save processed data
df_clean.to_csv(f'{OUTPUT_PATH}sensor_logs_cleaned.csv', index=False)
df_normalized.to_csv(f'{OUTPUT_PATH}sensor_logs_normalized.csv', index=False)

print("=== Data Preprocessing Complete ===")
print(f"Cleaned data saved to: {OUTPUT_PATH}sensor_logs_cleaned.csv")
print(f"Normalized data saved to: {OUTPUT_PATH}sensor_logs_normalized.csv")
print(f"\nFinal dataset shape: {df_clean.shape}")